# Context and regressors

Context is information you know about each trial, such as stimulus strength or the experimental condition. Superstats can use the same trial information in two ways.

- **Simulator context** supplies inputs directly to the simulator.
- **Design context** supplies predictors to formulas that compute simulator parameters.

The examples start with fixed trial inputs, then show how to generate fresh inputs for each dataset. The final examples evaluate formulas directly so you can see what each predictor changes.

In [ ]:
import numpy as np
import superstats as sup

## Simulator context

In the Racing Diffusion Model (RDM), evidence for two response options races toward a threshold. The first option to reach its threshold determines the choice. The trial input `boost_idx` selects which option gets a drift advantage, using indices `0` and `1`.

The fixed sequence alternates the boosted option across trials and repeats for both datasets. In the output, each row is a dataset and each column is a trial. The simulated choices can differ between datasets.

In [2]:
prior = sup.JointPrior(
    v_base=0.5,
    v_diff=1.5,
    a_base=0.6,
    a_diff=0.0,
    tau=0.2,
    sigma_diff=0.0,
)

model = sup.Model(
    prior=prior,
    simulator=sup.simulation.sample_rdm,
    context_simulator={"boost_idx": np.array([0, 1, 0, 1])},
    simulator_context=("boost_idx",),
    missing=None,
)

sample = model.sample(batch_size=2, num_steps=4)
print("Boosted option by dataset and trial")
print(sample["boost_idx"])
print("Simulated choice by dataset and trial")
print(sample["choice"])

Boosted option by dataset and trial
[[0 1 0 1]
 [0 1 0 1]]
Simulated choice by dataset and trial
[[0. 0. 1. 1.]
 [0. 0. 1. 1.]]


## Design context and a continuous predictor

Now let stimulus strength `x` change the drift contrast `v_diff`. The formula starts at `v_0 = 1.5` when `x = 0` and adds `b_x = 0.5` for each unit increase in `x`. For example, the first trial has `x = -1`, so its contrast is `1.0`.

Both kinds of context are used here. `x` changes the contrast through the formula, and `boost_idx` still tells the simulator which option gets the advantage. The next examples use the same model setup with a different source of context.

In [3]:
regression_prior = sup.JointPrior(
    v_base=0.5,
    v_0=1.5,
    b_x=0.5,
    a_base=0.6,
    a_diff=0.0,
    tau=0.2,
    sigma_diff=0.0,
)
regression_formula = sup.Formula(["v_diff = v_0 + b_x * x"])

fixed_context = {
    "x": np.array([-1.0, 0.0, 0.5, 1.0]),
    "boost_idx": np.array([0, 1, 0, 1]),
}

model = sup.Model(
    prior=regression_prior,
    simulator=sup.simulation.sample_rdm,
    formula=regression_formula,
    context_simulator=fixed_context,
    design_context=("x",),
    simulator_context=("boost_idx",),
    missing=None,
)

sample = model.sample(batch_size=2, num_steps=4)
print("Stimulus strength by trial")
print(sample["x"][0])
print("Drift contrast by dataset and trial")
print(sample["v_diff"][..., 0])

Stimulus strength by trial
[-1.   0.   0.5  1. ]
Drift contrast by dataset and trial
[[1.   1.5  1.75 2.  ]
 [1.   1.5  1.75 2.  ]]


## Simulate fresh context for each dataset

A fixed mapping repeats the same design for every dataset. To randomize the design, pass a function as `context_simulator`. Superstats passes it `batch_size` (how many datasets) and `num_steps` (trials per dataset). Each returned array has shape `(batch_size, num_steps)`, with one row of trial inputs per dataset.

Here each trial draws its own stimulus strength and boosted option. The model still uses `x` in the formula and routes `boost_idx` directly to the simulator.

In [4]:
rng = np.random.default_rng(42)


def generate_context(batch_size, num_steps):
    shape = (batch_size, num_steps)
    return {
        "x": rng.uniform(-1.0, 1.0, size=shape),
        "boost_idx": rng.integers(0, 2, size=shape),
    }


generated_model = sup.Model(
    prior=regression_prior,
    simulator=sup.simulation.sample_rdm,
    formula=regression_formula,
    context_simulator=generate_context,
    design_context=("x",),
    simulator_context=("boost_idx",),
    missing=None,
)
generated_sample = generated_model.sample(batch_size=2, num_steps=4)
print("Stimulus strength by dataset and trial")
print(generated_sample["x"].round(2))

Stimulus strength by dataset and trial
[[ 0.72  0.39 -0.81  0.95]
 [ 0.52  0.57 -0.74 -0.1 ]]


## Wrap a function that generates one dataset

Passing a function directly is equivalent to `sup.ContextSimulator(generate_context, is_batched=True)`. If your function generates one dataset at a time, wrap it with `is_batched=False`. The wrapper calls the function once per dataset with `num_steps` and stacks the results into batch-shaped arrays.

Call the wrapper's `sample()` method to inspect the trial inputs on their own, or pass the wrapper into the model to generate context and observations together.

In [5]:
def generate_one_context(num_steps):
    return {
        "x": rng.uniform(-1.0, 1.0, size=num_steps),
        "boost_idx": rng.integers(0, 2, size=num_steps),
    }


wrapped_context = sup.ContextSimulator(generate_one_context, is_batched=False)
context_draw = wrapped_context.sample(batch_size=2, num_steps=4)
print("Stimulus strength shape", context_draw["x"].shape)

wrapped_model = sup.Model(
    prior=regression_prior,
    simulator=sup.simulation.sample_rdm,
    formula=regression_formula,
    context_simulator=wrapped_context,
    design_context=("x",),
    simulator_context=("boost_idx",),
    missing=None,
)
wrapped_sample = wrapped_model.sample(batch_size=2, num_steps=4)
print("Drift contrast by dataset and trial")
print(wrapped_sample["v_diff"][..., 0].round(2))

Stimulus strength shape (2, 4)
Drift contrast by dataset and trial
[[1.04 1.15 1.68 1.74]
 [1.37 1.47 1.19 1.13]]


## A categorical predictor

Use `Categorical` to list the possible labels and choose a reference level. Here `neutral` uses the baseline `v_0`. The `C(...)` term adds `b_speed` on speed trials or `b_accuracy` on accuracy trials.

From here on, `formula.resolve()` returns the parameter values directly. Each output value corresponds to one trial in the order given by the context arrays.

In [6]:
categorical = {
    "condition": sup.Categorical(["neutral", "speed", "accuracy"], reference="neutral"),
}

formula = sup.Formula(
    ["v_diff = v_0 + C(condition, speed=b_speed, accuracy=b_accuracy)"],
    categorical=categorical,
)

formula.resolve(
    parameters={"v_0": 1.5, "b_speed": -0.3, "b_accuracy": 0.4},
    context={"condition": np.array(["neutral", "speed", "accuracy"])},
)["v_diff"]

array([1.5, 1.2, 1.9])

## A numeric 0/1 predictor

A group or condition already coded as `0` or `1` can go straight into the formula. A trial coded `0` uses the baseline `v_0`. A trial coded `1` adds `b_dummy`.

In [7]:
formula = sup.Formula(["v_diff = v_0 + b_dummy * dummy"])

formula.resolve(
    parameters={"v_0": 1.5, "b_dummy": 0.4},
    context={"dummy": np.array([0, 1, 0, 1])},
)["v_diff"]

array([1.5, 1.9, 1.5, 1.9])

## Continuous × continuous interaction

An interaction lets one predictor change the effect of another. Here the formula multiplies `x` and `z`, so the effect of `x` depends on `z`. On the middle trial, `x = 0` and the contrast stays at the baseline of `1.5`.

In [8]:
formula = sup.Formula(["v_diff = v_0 + b_interaction * x * z"])

formula.resolve(
    parameters={"v_0": 1.5, "b_interaction": 0.4},
    context={"x": np.array([-1.0, 0.0, 1.0]), "z": np.array([0.5, 1.0, 0.5])},
)["v_diff"]

array([1.3, 1.5, 1.7])

## Continuous × categorical interaction

The `C(...)` term picks the slope for each condition, then multiplies it by `x`. On the speed trial with `x = 0.5`, a slope of `-0.3` lowers the contrast from `1.5` to `1.35`. The reference condition has a slope of zero in this example.

In [9]:
formula = sup.Formula(
    ["v_diff = v_0 + C(condition, speed=b_speed, accuracy=b_accuracy) * x"],
    categorical=categorical,
)

formula.resolve(
    parameters={"v_0": 1.5, "b_speed": -0.3, "b_accuracy": 0.4},
    context={
        "condition": np.array(["neutral", "speed", "accuracy"]),
        "x": np.array([1.0, 0.5, 1.0]),
    },
)["v_diff"]

array([1.5 , 1.35, 1.9 ])

## Categorical × categorical interaction

The outer `C(...)` checks the group. Control trials keep the baseline `v_0`. For treatment trials, the inner `C(...)` picks the adjustment for the condition, so a speed trial gets `b_speed` and an accuracy trial gets `b_accuracy`.

In [10]:
formula = sup.Formula(
    ["v_diff = v_0 + C(group, treatment=C(condition, speed=b_speed, accuracy=b_accuracy))"],
    categorical={
        "condition": categorical["condition"],
        "group": sup.Categorical(["control", "treatment"], reference="control"),
    },
)

formula.resolve(
    parameters={"v_0": 1.5, "b_speed": -0.3, "b_accuracy": 0.4},
    context={
        "group": np.array(["control", "treatment", "treatment"]),
        "condition": np.array(["speed", "speed", "accuracy"]),
    },
)["v_diff"]

array([1.5, 1.2, 1.9])